# SNPObject

`SNPObject` is the central genotype container in `snputils`. This notebook builds a phased synthetic dataset with complete variant metadata, then demonstrates current object operations without relying on files outside the package.

## Create a phased SNPObject

`build_synthetic_snp_dataset()` creates a small `SNPObject` with genotype calls, sample IDs, population labels in `sample_fid`, sex labels, and variant metadata. Phased genotypes use shape `(n_snps, n_samples, 2)`, where the last dimension is the two haplotypes.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from snputils.datasets import build_synthetic_snp_dataset

RESULTS_DIR = Path("results/tutorials")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SEED = 20240520

snpobj = build_synthetic_snp_dataset(
    n_samples=6,
    n_snps=12,
    seed=SEED,
    n_populations=3,
    missing_rate=0.03,
    phased=True,
    chromosome="chr1",
)

snpobj

SNPObject(shape=(12, 6, 2), n_snps=12, n_samples=6, genotypes_shape=(12, 6, 2), calldata_lai_shape=None, calldata_gp_shape=None, has_variant_metadata=True, has_ancestry_map=False)

In [2]:
print("shape:", snpobj.shape)
print("n_samples:", snpobj.n_samples)
print("n_snps:", snpobj.n_snps)
print("unique chromosomes:", snpobj.unique_chrom)
print("keys:", snpobj.keys())

shape: (12, 6, 2)
n_samples: 6
n_snps: 12
unique chromosomes: ['chr1']
keys: ['genotypes', 'samples', 'variants_ref', 'variants_alt', 'variants_chrom', 'variants_cm', 'variants_filter_pass', 'variants_id', 'variants_pos', 'variants_qual', 'variants_info', 'calldata_lai', 'calldata_gp', 'ancestry_map', 'sample_fid', 'sample_sex']


## Access and update fields

Attributes can be accessed directly or with dictionary-style keys. Metadata arrays must stay aligned to the sample or variant axis.

In [3]:
print(snpobj.samples)
print(snpobj["variants_id"][:4])

renamed = snpobj.copy()
renamed.samples = np.array([f"sample_{i}" for i in range(renamed.n_samples)])
renamed.samples

['S01' 'S02' 'S03' 'S04' 'S05' 'S06']
['rs_syn_00001' 'rs_syn_00002' 'rs_syn_00003' 'rs_syn_00004']


array(['sample_0', 'sample_1', 'sample_2', 'sample_3', 'sample_4',
       'sample_5'], dtype='<U8')

## Dosage and strand handling

`dosage()` returns alternate-allele dosage while preserving missing calls. `sum_strands()` converts phased allele calls to a 2D dosage matrix.

In [4]:
alt_dosage = snpobj.dosage("ALT")
summed = snpobj.sum_strands()

print("ALT dosage shape:", alt_dosage.shape)
print("Summed genotype shape:", summed.genotypes.shape)
print(summed.genotypes[:3])

ALT dosage shape: (12, 6)
Summed genotype shape: (12, 6)
[[2 2 0 2 2 1]
 [1 2 1 1 2 1]
 [0 0 0 1 2 0]]


## Variant and sample filters

Filtering returns a copy unless `inplace=True`. You can filter variants by chromosome, position, index, or a boolean mask, and filter samples by ID or index.

In [5]:
chr1 = snpobj.filter_variants(chrom="chr1")
ordered_samples = snpobj.filter_samples(samples=["S05", "S01"], reorder=True)
complete = snpobj.filter_complete_genotypes()
polymorphic = snpobj.filter_polymorphic_variants()

print("chr1 variants:", chr1.variants_id.tolist())
print("ordered samples:", ordered_samples.samples.tolist())
print("complete variants:", complete.n_snps)
print("polymorphic variants:", polymorphic.n_snps)

chr1 variants: ['rs_syn_00001', 'rs_syn_00002', 'rs_syn_00003', 'rs_syn_00004', 'rs_syn_00005', 'rs_syn_00006', 'rs_syn_00007', 'rs_syn_00008', 'rs_syn_00009', 'rs_syn_00010', 'rs_syn_00011', 'rs_syn_00012']
ordered samples: ['S05', 'S01']
complete variants: 10
polymorphic variants: 12


## Chromosome naming

The object can detect and convert chromosome formats. This is useful before merging datasets that mix `chr1` and `1` naming conventions.

In [6]:
print("before:", snpobj.detect_chromosome_format(), snpobj.variants_chrom[:3])
no_chr = snpobj.convert_chromosome_format("chr", "plain")
print("after:", no_chr.detect_chromosome_format(), no_chr.variants_chrom[:3])

before: chr ['chr1' 'chr1' 'chr1']
after: plain ['1' '1' '1']


## Allele frequencies

Allele frequencies can be computed for the whole cohort or by sample label. `sample_fid` is a natural source of population labels for small examples.

In [7]:
af_by_pop, called_by_pop = snpobj.allele_freq(
    sample_labels=snpobj.sample_fid,
    return_counts=True,
    as_dataframe=True,
)

display(af_by_pop.head())
display(called_by_pop.head())

,POP_A,POP_B,POP_C
0,1.00,1.0,0.666667
1,0.50,1.0,0.500000
2,0.25,0.5,0.000000
3,0.50,0.5,0.500000
4,1.00,0.5,0.500000


,POP_A,POP_B,POP_C
0,4,4,3
1,4,4,4
2,4,4,4
3,4,4,4
4,4,4,4


## Local ancestry on a SNPObject

`calldata_lai` can be attached directly to SNPs. The array shape mirrors phased genotypes: `(n_snps, n_samples, 2)`.

In [8]:
snp_with_lai = snpobj.copy()
rng = np.random.default_rng(SEED)
snp_with_lai.calldata_lai = rng.integers(0, 3, size=snpobj.genotypes.shape, dtype=np.int8)
snp_with_lai.ancestry_map = {"0": "AFR", "1": "EUR", "2": "AMR"}

af_afr, called_afr = snp_with_lai.allele_freq(ancestry=0, return_counts=True)
print("AFR-specific AF:", np.round(af_afr[:6], 3))
print("called haplotypes:", called_afr[:6])

AFR-specific AF: [0.75  0.75  0.286 0.667 0.857 0.   ]
called haplotypes: [4 4 7 3 7 2]
